In [1]:
# !pip install langchain_community

In [2]:
import pymupdf4llm

from sentence_transformers import SentenceTransformer
from langchain.text_splitter import RecursiveCharacterTextSplitter
import chromadb
import ollama
import time
from collections import deque

import time

c:\Users\mhseo\anaconda3\envs\pdf_bot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_NAME = 'exaone3.5:2.4b'

In [4]:
# 기존 설치 확인
ollama.list()

ListResponse(models=[Model(model='exaone3.5:2.4b', modified_at=datetime.datetime(2026, 3, 11, 14, 50, 43, 72032, tzinfo=TzInfo(+09:00)), digest='13644fc3d28eaaff502f903406496cad15df31e367cb7be988c132def104ba42', size=1644933401, details=ModelDetails(parent_model='', format='gguf', family='exaone', families=['exaone'], parameter_size='2.7B', quantization_level='Q4_K_M'))])

In [5]:
# exaone3.5:2.4b 설치
ollama.pull('exaone3.5:2.4b')

ProgressResponse(status='success', completed=None, total=None, digest=None)

### exaone3.5 테스트용 

In [6]:
%%time

response = ollama.chat(model=MODEL_NAME, messages=[
  {
    'role': 'user',
    'content': '청년미래적금에 대해 설명하고, 2026년에 시행하는 청년 정책에 대해 설명해줘.',
  },
])
print(response['message']['content'])

time.sleep(1)

### 청년미래적금 개요

**청년미래적금**은 한국 정부가 청년층을 위한 저축 지원 정책의 일환으로 도입한 제도입니다. 주요 목적과 특징은 다음과 같습니다:

1. **대상**: 만 18세에서 39세 사이의 청년들이 주로 대상입니다.
2. **가입 기간**: 일반적으로 평생 저축을 목표로 하며, 정기적으로 저축 금액을 적립할 수 있도록 설계되었습니다. 초기 가입 시 일정 금액을 설정하고, 이후 일정 기간 동안 일정 비율로 증액되거나 일정 금액을 매월 또는 매년 추가로 납입하도록 권장됩니다.
3. **지원 내용**:
   - **정부 지원**: 정부가 일정 비율의 저축 금액을 보조하여 실질적인 저축 부담을 줄입니다.
   - **세제 혜택**: 저축액에 대한 세금 감면 혜택이 제공되어 저축 동기 부여에 도움을 줍니다.
   - **금융 교육**: 정기적인 금융 교육 프로그램을 통해 청년들의 재무 관리 능력 향상을 지원합니다.
4. **목표**: 청년층의 경제적 안정성 강화와 장기적인 재정 건전성 확보를 목표로 합니다.

### 2026년 청년 정책 전망

2026년을 앞둔 현재, 정부는 청년들의 미래를 위한 다양한 정책 개선 및 신규 정책 도입에 대한 관심을 높이고 있습니다. 주요 방향과 예상되는 정책들은 다음과 같습니다:

1. **확대된 청년 미래적금**:
   - **보완 및 강화**: 기존 청년미래적금 제도를 더욱 보완하여 더 많은 청년들이 쉽게 접근할 수 있도록 개선될 가능성이 큽니다. 예를 들어, 더 유연한 저축 기간 설정이나 다양한 연령대를 포함한 포괄적인 대상 확대가 예상됩니다.
   - **증가된 지원 비율**: 정부의 저축 지원 비율이 더 높아져 청년들의 저축 부담을 더욱 경감시킬 것으로 보입니다.

2. **교육 및 취업 지원 강화**:
   - **맞춤형 교육 프로그램**: IT 기술, 혁신 산업 분야 등 미래 경쟁력을 갖춘 직업군에 초점을 맞춘 맞춤형 교육 프로그램 확대.
   - **취업 연계 서비스**: 스타트업 지원, 창업 컨설팅,

In [7]:
ollama.list()

ListResponse(models=[Model(model='exaone3.5:2.4b', modified_at=datetime.datetime(2026, 3, 11, 16, 14, 6, 803725, tzinfo=TzInfo(+09:00)), digest='13644fc3d28eaaff502f903406496cad15df31e367cb7be988c132def104ba42', size=1644933401, details=ModelDetails(parent_model='', format='gguf', family='exaone', families=['exaone'], parameter_size='2.7B', quantization_level='Q4_K_M'))])

### pdf 파일 경로 및 텍스트 분할

In [8]:
# PDF 파일 경로
file_path = "./pdf_folder/251226보도자료별첨 제2차 청년정책 기본계획26-30국무조정실.pdf"

# 로컬 인공지능 언어 모델 
model = MODEL_NAME

def load_pdf(file_path):
    print("PDF파일 로드", end = '')
    start = time.time()
    pdf_data = pymupdf4llm.to_markdown(file_path)
    text = "".join(pdf_data)  # 페이지별 텍스트를 하나의 문자열로 결합
    time.sleep(1)
    print(f' : {time.time() - start:.4f} sec')
    return text

def split_text(text):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    return text_splitter.split_text(text)

### 임베딩 및 벡터 DB 저장

In [9]:
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="rag_collection",  metadata={"hnsw:space": "cosine"})

start = time.time() # 시작

raw_text = load_pdf(file_path)

print("임베딩", end = '')
chunks = split_text(raw_text)
embeddings = embedder.encode(chunks, convert_to_tensor=False)

for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    collection.add(
        ids=[f"chunk_{i + 1}"],
        embeddings=[embedding.tolist()],
        metadatas=[{"text": chunk}],
    )

time.sleep(1)

print(f' : {time.time() - start:.4f} sec')

PDF파일 로드 : 20.1828 sec
임베딩 : 36.9490 sec


### 대화 히스토리 기록 클래스

In [10]:
class Message_manager:
    def __init__(self):
        self._system_msg = {"role": "system", "content": ""}
        self.queue = deque(maxlen=10)  # 최대 10개 대화 저장

    def create_msg(self, role, content):
        return {"role": role, "content": content}

    def system_msg(self, content):
        self._system_msg = self.create_msg("system", content)

    def append_msg(self, content):
        msg = self.create_msg("user", content)
        self.queue.append(msg)

    def get_chat(self):
        return [self._system_msg] + list(self.queue)

    def set_retrived_docs(self, docs):
        self.retrieved_docs = docs

    def append_msg_by_assistant(self, content):
        msg = self.create_msg("assistant", content)
        self.queue.append(msg)

    def generate_prompt(self, retrieved_docs):

        docs = "\n".join(retrieved_docs)

        prompt = [msgManager._system_msg,{
            "role": "system",
            "content": f"문서 내용: {docs}\n질문에 대한 답변은 문서 내용을 기반으로 정확히 제공하시오.",
        }] + list(msgManager.queue)

        return prompt


msgManager = Message_manager() # 객체 생성

msgManager.system_msg("""
    1. 모든 답변은 반드시 'user'가 제공한 'content'와 'system' 메시지에 포함된 '문서 내용'만을 바탕으로 작성한다.
    2. 문서 내용에 명시되지 않은 정보나 모델이 기존에 학습한 외부 지식은 절대 답변에 포함하지 않는다.
    3. 질문에 대한 답이 문서에 존재하지 않을 경우, "제시된 문서 내에서 해당 정보를 찾을 수 없습니다"라고 명확히 답변한다.
    4. 답변의 신뢰도를 위해 문서의 어느 부분(장, 절, 혹은 핵심 키워드)을 참고했는지 가급적 명시한다.
    5. 개행은 문장이 끝날 때와 서로 다른 주제나 항목을 구분할 때만 사용하며, 가독성을 저해하는 불필요한 개행은 금지한다.
    6. 전문적인 용어는 문서에 사용된 그대로를 사용하되, 문맥상 이해가 필요한 경우에만 보충 설명을 덧붙인다.                   
"""                     
)

### 질문처리 함수

In [11]:
def retrieve_docs(query, collection, embedder, top_k=2):
    query_embedding = embedder.encode(query, convert_to_tensor=False)
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k)
    
    if not results["metadatas"]:  # 검색 결과가 없는 경우
        return ["관련 문서를 찾을 수 없습니다."]
    
    docs = [doc["text"] for doc in results["metadatas"][0]]
    return docs

def generate_answer(query, retrieved_docs, conversation_history):
    # 이전 대화 기록 추가 (최대 10개)
    msgManager.append_msg(query)
    
    # 스트리밍으로 답변 생성
    print("답변: ", end="")
    full_answer = ""
    
    msg = msgManager.generate_prompt(retrieved_docs)

    for response in ollama.chat(model=model, messages=msg, stream=True):
        chunk = response["message"]["content"]
        print(chunk, end="")
        full_answer += chunk
    
    # 'assistant' 답변 추가, 기록 남기기
    conversation_history.append({
        "role": "assistant",
        "content": full_answer
    })

    return full_answer

### 대화형 루프

In [12]:
def chat_loop():
    print("RAG 챗봇 시작! 질문 입력 (종료하려면 'exit' 입력):")
    
    conversation_history = []   # 대화 기록용 리스트
    conversation_log = []      # 사용자, 챗봇, 참고 문서 기록용

    while True:
        query = input("> ")
        print(query)
        print('\n')
        
        start = time.time()
        if query.lower() == "exit":
            print("챗봇 종료!")
            break

        retrieved_docs = retrieve_docs(query, collection, embedder, top_k=2)
        answer = generate_answer(query, retrieved_docs, conversation_history)
        msgManager.append_msg_by_assistant(answer)
        
        time.sleep(1)
        
        # Log 남기기
        conversation_log.append({
            "사용자": query,
            "챗봇": answer,
            "참고 문서": retrieved_docs
        })
        
    return conversation_log

In [13]:
q_1 = "구직단념 청년을 위해 시행하는 정책이 무엇이 있어?"

In [14]:
conversation_log_data = chat_loop()

RAG 챗봇 시작! 질문 입력 (종료하려면 'exit' 입력):
구직단념 청년을 위해 시행하는 정책이 무엇이 있어?


답변: 구직단념 청년을 위한 정책은 다음과 같이 주요 세 가지로 구성되어 있습니다:

1. **청년 일자리 첫걸음 플랫폼**:
   - **개요**: 미취업 대학 졸업생과 군 복무를 마친 청년들을 대상으로 고용보험 데이터베이스와 연결된 플랫폼을 구축합니다.
   - **특징**: 개인정보 제공 동의를 통해 플랫폼에 참여하여 장기 미취업 위험군을 선제적으로 발굴하고 맞춤형 취업 지원을 받을 수 있습니다. 비대면 참여가 가능하며, 밀착형 멘토링 등 다양한 형태의 지원을 제공합니다.

2. **‘쉬고 있는 청년’ 특화 일경험 프로그램**:
   - **개요**: 사회 연대 경제, 공공 부문을 통해 청년들이 점진적으로 사회 적응력을 키우고 경력을 형성할 수 있도록 지원합니다.
   - **특징**: 비대면 참여 방식을 유연하게 조정하고, 밀착형 멘토링 등을 통해 청년들의 적응과 재취업 준비를 돕습니다.

3. **청년도전지원사업**:
   - **개요**: 구직 단념 청년을 위한 맞춤형 프로그램으로, 상담, 역량 강화, 구직 의욕 고취 등을 지원합니다.
   - **특징**: 청년들에게 직접 참여 수당을 지원하며, 금액은 50만원에서 250만원까지 다양하게 제공됩니다. 이를 통해 구직 단념 청년들이 적극적으로 재취업을 시도할 수 있도록 돕습니다.

이러한 정책들은 청년들이 재취업 과정에서 겪는 어려움을 완화하고, 다양한 지원을 통해 경제 활동 참여를 촉진하는 것을 목표로 하고 있습니다.exit


챗봇 종료!


In [15]:
# !pip install pandas
# !pip install numpy

In [19]:
import pandas as pd

In [20]:
conversation_log_df = pd.DataFrame(conversation_log_data)
conversation_log_df

,사용자,챗봇,참고 문서
0,구직단념 청년을 위해 시행하는 정책이 무엇이 있어?,구직단념 청년을 위한 정책은 다음과 같이 주요 세 가지로 구성되어 있습니다:\n\n...,[-----\n\n2 제2차 청년정책 기본계획 체계 및 핵심과제\n\n\n\n\n\...


In [21]:
conversation_log_data

[{'사용자': '구직단념 청년을 위해 시행하는 정책이 무엇이 있어?',
  '챗봇': '구직단념 청년을 위한 정책은 다음과 같이 주요 세 가지로 구성되어 있습니다:\n\n1. **청년 일자리 첫걸음 플랫폼**:\n   - **개요**: 미취업 대학 졸업생과 군 복무를 마친 청년들을 대상으로 고용보험 데이터베이스와 연결된 플랫폼을 구축합니다.\n   - **특징**: 개인정보 제공 동의를 통해 플랫폼에 참여하여 장기 미취업 위험군을 선제적으로 발굴하고 맞춤형 취업 지원을 받을 수 있습니다. 비대면 참여가 가능하며, 밀착형 멘토링 등 다양한 형태의 지원을 제공합니다.\n\n2. **‘쉬고 있는 청년’ 특화 일경험 프로그램**:\n   - **개요**: 사회 연대 경제, 공공 부문을 통해 청년들이 점진적으로 사회 적응력을 키우고 경력을 형성할 수 있도록 지원합니다.\n   - **특징**: 비대면 참여 방식을 유연하게 조정하고, 밀착형 멘토링 등을 통해 청년들의 적응과 재취업 준비를 돕습니다.\n\n3. **청년도전지원사업**:\n   - **개요**: 구직 단념 청년을 위한 맞춤형 프로그램으로, 상담, 역량 강화, 구직 의욕 고취 등을 지원합니다.\n   - **특징**: 청년들에게 직접 참여 수당을 지원하며, 금액은 50만원에서 250만원까지 다양하게 제공됩니다. 이를 통해 구직 단념 청년들이 적극적으로 재취업을 시도할 수 있도록 돕습니다.\n\n이러한 정책들은 청년들이 재취업 과정에서 겪는 어려움을 완화하고, 다양한 지원을 통해 경제 활동 참여를 촉진하는 것을 목표로 하고 있습니다.',
  '참고 문서': ['-----\n\n2 제2차 청년정책 기본계획 체계 및 핵심과제\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n               - 14 \n\n-----\n\n                     - 15 \n\n-----\n\n                     - 16 \n\n-----\n\n|일자리 분야|재학생|▸대학 일자리